# Catchment area analysis of supermarkets

## Libraries and settings

In [1]:
# Libraries
import os
import time
import glob
import json
import folium
import requests
import platform
import numpy as np
import pandas as pd
import geopandas as gdp
import matplotlib.pyplot as plt
from openrouteservice import client
from IPython.display import clear_output

# Read OpenRouteService API key
with open(file='ors_token.txt', mode='r') as file:
    api_key = file.read()
# print(api_key)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Create isochrone for a defined location

In [2]:
# Client settings
ors = client.Client(key=api_key)

# Settings
transport = 'driving-car'  # 'driving-car', 'foot-walking', 'cycling-regular'
traveltime = 15  # in minutes

# Location (lat, lon)
location = (47.37445, 8.54104)

# Parameters for server-request
params_iso = {
    'locations': [(location[1], location[0])],  # (lon, lat)
    'profile': transport,
    'range': [traveltime * 60],  # in seconds
    'attributes': ['total_pop']
}

# Server request
isochrone = ors.isochrones(**params_iso)

# Set up folium map
map = folium.Map(tiles='OpenStreetMap', 
                 location=location, 
                 zoom_start=11)

# Add isochrone to the map
folium.GeoJson(isochrone).add_to(map)

# Display the map in a Jupyter Notebook (if applicable)
map

## Import supermarket data

In [3]:
# Import supermarket data
df_orig = pd.read_csv('supermarkets_data_enriched.csv', 
                      sep=',',
                      encoding='utf-8')[['id',
                                         'lat',
                                         'lon',
                                         'brand',
                                         'bfs_number',
                                         'bfs_name',
                                         'addr:housenumber',
                                         'addr:postcode',
                                         'geometry']]

# Subset, only supermarkets with complete address
df = df_orig.dropna()
print(df.shape)
df.head()

(967, 9)


,id,lat,lon,brand,bfs_number,bfs_name,addr:housenumber,addr:postcode,geometry
1,280130028,47.155492,9.039666,ALDI,3315,Schänis,3,8718.0,POINT (9.0396663 47.1554921)
2,6122906632,47.158959,9.044477,Denner,3315,Schänis,14,8718.0,POINT (9.0444769 47.1589589)
4,36726161,47.226191,8.980329,Migros,3339,Uznach,25,8730.0,POINT (8.9803292 47.2261912)
6,7389830915,47.225056,8.970650,Denner,3339,Uznach,15,8730.0,POINT (8.9706499 47.2250563)
7,39947904,47.376732,8.542161,Coop,261,Zürich,1,8001.0,POINT (8.5421608 47.3767316)


## Create isochrone for a single supermarket

In [4]:
# Settings
municip = 'Adliswil'
transport = 'driving-car'  # 'driving-car', 'foot-walking', 'cycling-regular'
traveltime = 15

# Client settings
ors = client.Client(key=api_key)

# Set up supermarket dictionary with a single supermarket
df_sub = df.loc[df['bfs_name'] == municip].iloc[0]
supermarkets = {df_sub['brand']: {'location': [df_sub['lon'], 
                                               df_sub['lat']]}}
print(supermarkets)

# Set up folium map
map = folium.Map(tiles='OpenStreetMap', 
                 location=([df_sub['lat'], 
                            df_sub['lon']]), 
                 zoom_start=11)

# Parameters for server-request
params_iso = {'profile': transport,
              'range': [traveltime*60],
              'attributes': ['total_pop']}

# Server request
for name, apt in supermarkets.items():
    
    # Add coords to request parameters
    params_iso['locations'] = [apt['location']]
    
    # Perform isochrone request
    apt['iso'] = ors.isochrones(**params_iso)
    
    # Add GeoJson to map
    folium.features.GeoJson(apt['iso']).add_to(map)
    
    # Save GeoJson as file
    # with open(f'{name}.json', 'w') as f:
        # f.write(json.dumps(apt['iso']))

    # Reverse coords due to weird folium lat/lon syntax
    folium.map.Marker(list(reversed(apt['location'])),
                      icon = folium.Icon(color='green',
                                         icon_color='#cc0000',
                                         icon='home',
                                         prefix="fa"),
                      popup = name
                      ).add_to(map)

# Plot map
map

{'Denner': {'location': [np.float64(8.5218139), np.float64(47.3203453)]}}


## Getting the number of residents in the isochrone area

In [5]:
# Save the data as pandas data frame
data = pd.DataFrame(apt['iso']["features"])

# Create data frame from column 'properties'
df_pop = pd.DataFrame(data.loc[0, 'properties'])
val = df_pop['total_pop'][0]
print('Number of residents in isochrone area:', f'{val:,.0f}')

Number of residents in isochrone area: 159,339


## Estimating the purchasing power in the isochrone area

In [6]:
# Simple assumtions:
# 500 CHF per resident and month
# Residents / employee ratio = 2:1
# 20 CHF per employee and working day

# Result
val = ((df_pop['total_pop'][0] * 500) + (df_pop['total_pop'][0] * 0.5 * 20 * 20)) / 10**6
print('Purchasing power per month:',
      f'{val:.1f}',
     'Mio. CHF')

Purchasing power per month: 111.5 Mio. CHF


## Automization of the analysis for multiple of supermarkets

In [7]:
# Client settings
ors = client.Client(key=api_key)

# Parameters for server-request
params_iso = {'profile': 'driving-car',
              'range': [15*60],
              'attributes': ['total_pop']}

# Df to store the data
pop_out = []

try:
    for i in range(0, 10):
        
        print('Preparing isochrone for supermarkt', df[['id', 
                                                        'brand', 
                                                        'bfs_name', 
                                                        'addr:housenumber',
                                                        'addr:postcode']].iloc[i])
        
        supermarkets = {df['brand'].iloc[i]: {'location': [df['lon'].iloc[i],
                                                           df['lat'].iloc[i]]}}
        # Server request
        for name, apt in supermarkets.items():
            
            # Add coords to request parameters
            params_iso['locations'] = [apt['location']]
            
            # Perform isochrone request
            apt['iso'] = ors.isochrones(**params_iso)
            time.sleep(1)
            
            # Save the data as pandas data frame
            data = pd.DataFrame(apt['iso']["features"])
            
            # Create data frame from column 'properties'
            df_pop = pd.DataFrame(data.loc[0, 'properties'])
            
            # Write values
            pop_out.append([df['id'].iloc[i], 
                            df['brand'].iloc[i],
                            df['bfs_number'].iloc[i], 
                            df['bfs_name'].iloc[i], 
                            df['addr:housenumber'].iloc[i],
                            df['addr:postcode'].iloc[i],
                            df_pop['total_pop'][0],
                            ((df_pop['total_pop'][0] * 500) + (df_pop['total_pop'][0] * 0.5 * 20 * 20)) / 10**6])
            clear_output(wait=True)
            
except:
    pop_out.append([df['id'].iloc[i], 
                    df['brand'].iloc[i],
                    df['bfs_number'].iloc[i], 
                    df['bfs_name'].iloc[i], 
                    df['addr:housenumber'].iloc[i],
                    df['addr:postcode'].iloc[i],
                    np.NaN,
                    np.NaN])
    clear_output(wait=True)

# Convert the list to a DataFrame
columns = ['id', 'brand', 'bfs_number', 'bfs_name', 'addr:housenumber', 'addr:postcode', 'total_pop', 'purchasing_power']
df_result = pd.DataFrame(pop_out, columns=columns)

# Display the DataFrame
df_result


,id,brand,bfs_number,bfs_name,addr:housenumber,addr:postcode,total_pop,purchasing_power
0,280130028,ALDI,3315,Schänis,3,8718.0,79066.0,55.3462
1,6122906632,Denner,3315,Schänis,14,8718.0,77217.0,54.0519
2,36726161,Migros,3339,Uznach,25,8730.0,62043.0,43.4301
3,7389830915,Denner,3339,Uznach,15,8730.0,62221.0,43.5547
4,39947904,Coop,261,Zürich,1,8001.0,522620.0,365.8340
5,48932835,Migros,261,Zürich,7,8004.0,516362.0,361.4534
6,75749133,ALDI,261,Zürich,81,8038.0,316376.0,221.4632
7,79977755,Coop,261,Zürich,15,8038.0,309106.0,216.3742
8,83330862,Migros,261,Zürich,3,8038.0,318664.0,223.0648
9,119249170,Migros,261,Zürich,31-35,8001.0,522116.0,365.4812


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [8]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-13 09:54:16
Python Version: 3.12.14
-----------------------------------
